# ML-08 -- Capstone Modeling Lane

This notebook trains two ML models (Logistic Regression and Random Forest) on the June 2026 warehouse dataset and compares them against the hand-coded rule baseline from W04. All evaluation uses the same grouped client split, same features, and same metrics for an honest apples-to-apples comparison.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `training-honest-models/SKILL.md`.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

---

### Method: Logistic Regression --> Random Forest (readable --> stronger)

Our question is a **binary classification** problem: "Is this page a CTR underperformance opportunity?" (`is_opportunity = 0 or 1`). We also use predicted probabilities to **rank** pages for a priority queue (Precision@K).

Following the skill's guidance:
1. **Logistic Regression** first -- transparent, readable coefficients, fast to train. Establishes a learned baseline.
2. **Random Forest** second -- handles non-linear relationships (e.g., position tier thresholds) without manual feature engineering. Stronger but less interpretable.

We compare both against the hand-coded rule baseline from W04 on the **same split and same metrics**.

### Honest Feature Vector (5 safe features)

From our W03 leakage audit, we use only features that do NOT leak into the label:
- `log_impressions` -- log1p(total search views)
- `avg_position` -- weighted SERP rank
- `engagement_rate` -- GA4 engaged session %
- `word_count` -- page length in words
- `has_ga4_data` -- whether GA4 tracking is active (0/1)

**Banned features:** `observed_ctr` (leaks into label), `ctr_gap` (directly defines label).

In [5]:
# == Cell 1: Connect + Build Feature Matrix + Grouped Client Split ==
import duckdb
import os, sys
import pandas as pd
import numpy as np
import pathlib, getpass
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

SEED = 42
np.random.seed(SEED)

# Load HF_TOKEN from .env
_env = pathlib.Path(os.getcwd()).resolve()
for _ in range(5):
    _ep = _env / '.env'
    if _ep.exists():
        for _line in _ep.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith('#') and '=' in _line:
                _k, _v = _line.split('=', 1)
                os.environ.setdefault(_k.strip(), _v.strip())
        break
    _env = _env.parent

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF_TOKEN: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':      f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':      f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}
print('DuckDB connected.')

feature_vector_q = f"""
WITH monthly_agg_raw AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)        AS total_impressions,
        SUM(f.gsc_clicks)             AS total_clicks,
        CASE WHEN SUM(f.gsc_impressions) > 0
             THEN CAST(SUM(f.gsc_clicks) AS DOUBLE) / SUM(f.gsc_impressions) * 100.0
             ELSE 0.0
        END AS observed_ctr,
        CASE WHEN SUM(f.gsc_impressions) > 0
             THEN SUM(f.gsc_avg_position * f.gsc_impressions) / SUM(f.gsc_impressions)
             ELSE 0.0
        END AS avg_position,
        CASE WHEN SUM(f.ga4_sessions) > 0
             THEN CAST(SUM(f.ga4_engaged_sessions) AS DOUBLE) / SUM(f.ga4_sessions) * 100.0
             ELSE 0.0
        END AS engagement_rate,
        MAX(CASE WHEN f.ga4_data_available = TRUE THEN 1 ELSE 0 END) AS has_ga4_data,
        ANY_VALUE(d.word_count) AS word_count
    FROM {TABLES['fact_daily_sample']} f
    LEFT JOIN {TABLES['dim_content']} d ON f.content_hash_id = d.content_hash_id
    WHERE f.month = '2026-06'
    GROUP BY f.client_hash_id, f.content_hash_id
    HAVING total_impressions >= 500 AND avg_position > 0
),
monthly_agg AS (
    SELECT m.*,
        CASE
            WHEN m.avg_position <= 3  THEN 'pos_1_3'
            WHEN m.avg_position <= 10 THEN 'pos_4_10'
            WHEN m.avg_position <= 20 THEN 'pos_11_20'
            WHEN m.avg_position <= 50 THEN 'pos_21_50'
            ELSE 'pos_51_plus'
        END AS position_tier
    FROM monthly_agg_raw m
)
SELECT m.*, t.tier_median_ctr
FROM monthly_agg m
LEFT JOIN (
    SELECT position_tier, MEDIAN(observed_ctr) AS tier_median_ctr
    FROM monthly_agg
    GROUP BY position_tier
) t ON m.position_tier = t.position_tier
"""

df = con.sql(feature_vector_q).df()
df['word_count'] = df['word_count'].fillna(0)
df['log_impressions'] = np.log1p(df['total_impressions'])
df['ctr_gap'] = df['tier_median_ctr'] - df['observed_ctr']
df['is_opportunity'] = ((df['ctr_gap'] > 0) & (df['total_impressions'] >= 1000)).astype(int)

# ---- Honest feature vector ----
FEATURES = ['log_impressions', 'avg_position', 'engagement_rate', 'word_count', 'has_ga4_data']
X = df[FEATURES].values
y = df['is_opportunity'].values
groups = df['client_hash_id'].values

# ---- Grouped client split (80/20) ----
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f'Total pages: {len(df):,}')
print(f'Train: {len(train_idx):,} pages ({len(train_idx)/len(df)*100:.1f}%)')
print(f'Test:  {len(test_idx):,} pages ({len(test_idx)/len(df)*100:.1f}%)')
print(f'Train clients: {len(set(groups[train_idx]))} | Test clients: {len(set(groups[test_idx]))}')
print(f'Train label rate: {y_train.mean():.4f} | Test label rate: {y_test.mean():.4f}')
print(f'Features: {FEATURES}')
print(f'Random seed: {SEED}')

DuckDB connected.
Total pages: 52,766
Train: 40,697 pages (77.1%)
Test:  12,069 pages (22.9%)
Train clients: 35 | Test clients: 9
Train label rate: 0.3756 | Test label rate: 0.2053
Features: ['log_impressions', 'avg_position', 'engagement_rate', 'word_count', 'has_ga4_data']
Random seed: 42


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

---

### Grouped Client Split (`GroupShuffleSplit`)

We use **GroupShuffleSplit grouped by `client_hash_id`** with an 80/20 train/test ratio.

**Why this split is honest:**
- Pages from the **same client** share hidden similarities (same website structure, same industry, same GA4 setup).
- If we allowed the same client's pages in both train and test, the model could memorize client-level patterns instead of learning general page-level signals.
- By putting entire clients into either train OR test (never both), we test whether the model generalizes to **unseen clients** -- which is the real-world deployment scenario.

**Empirical split:** Train = 40,697 pages (35 clients) | Test = 12,069 pages (9 clients).

**Important observation:** The test label rate (20.5%) is lower than the train label rate (37.6%). This is **expected and honest** -- different clients have different opportunity profiles, and the grouped split reveals this real-world variance. A random split would hide it.

**Random seed = 42** for full reproducibility.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

---

### Comparison Table: Baseline Rule vs Logistic Regression vs Random Forest

All three systems are evaluated on the **same test set** (grouped client holdout) using **Precision@K** and **ROC AUC**.

### Key Finding: The Rule Baseline Wins at Precision@K

The hand-coded rule baseline achieves **100% Precision@K** across all K values, while both ML models trail significantly (LR peaks at 85%, RF at 55%).

**Why does the rule baseline win?** This is the most important insight from this notebook:
- The rule baseline **directly uses `ctr_gap`** in its formula (`baseline_score = visible * clickable * underperforming * ctr_gap * log_impressions`).
- `ctr_gap` is closely tied to the label definition (`is_opportunity = ctr_gap > 0 AND impressions >= 1000`).
- The ML models are **banned from using `ctr_gap`** (it leaks!) and must infer opportunity from only 5 honest features.
- This is an honest finding: the rule is hard to beat precisely because it uses the label-defining metric.

**ROC AUC tells a different story:** Rule baseline (0.814) > RF (0.796) > LR (0.750). The RF comes within 0.018 AUC of the rule baseline using only honest features -- a strong result given the handicap.

In [6]:
# == Cell 2: Train LR + RF + Compare vs Baseline ==

# ---- 1. Rule Baseline Score (same formula as W04) ----
df_test = df.iloc[test_idx].copy()
visible = (df_test['total_impressions'] >= 1000).astype(int)
clickable = (df_test['avg_position'] <= 20).astype(int)
underperforming = (df_test['ctr_gap'] > 0).astype(int)
df_test['baseline_score'] = visible * clickable * underperforming * df_test['ctr_gap'] * df_test['log_impressions']
baseline_scores = df_test['baseline_score'].values

# ---- 2. Logistic Regression (needs scaled features) ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(random_state=SEED, max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_probs = lr.predict_proba(X_test_scaled)[:, 1]

# ---- 3. Random Forest ----
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

# ---- Evaluation helpers ----
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = y_test.mean()

# ---- ROC AUC ----
# Baseline scores may have zero variance edge cases; handle gracefully
try:
    baseline_auc = roc_auc_score(y_test, baseline_scores)
except ValueError:
    baseline_auc = float('nan')
lr_auc = roc_auc_score(y_test, lr_probs)
rf_auc = roc_auc_score(y_test, rf_probs)

# ---- Print Comparison Table ----
print('=' * 90)
print('COMPARISON TABLE: Baseline Rule vs Logistic Regression vs Random Forest')
print('=' * 90)
print(f'Test set: {len(y_test):,} pages | Base rate (is_opportunity): {base_rate:.4f} ({base_rate*100:.1f}%)')
print()

header = f'{"Metric":<20} {"Base Rate":>10} {"Rule Baseline":>14} {"Logistic Reg":>14} {"Random Forest":>14}'
print(header)
print('-' * len(header))

for k in [10, 20, 50, 100, 200, 500]:
    if k <= len(y_test):
        p_base = base_rate
        p_rule = precision_at_k(baseline_scores, y_test, k)
        p_lr   = precision_at_k(lr_probs, y_test, k)
        p_rf   = precision_at_k(rf_probs, y_test, k)
        print(f'Precision@{k:<10d} {p_base:>9.1%} {p_rule:>13.1%} {p_lr:>13.1%} {p_rf:>13.1%}')

print(f'{"ROC AUC":<20} {0.500:>10.3f} {baseline_auc:>14.3f} {lr_auc:>14.3f} {rf_auc:>14.3f}')
print()
print('Note: All models evaluated on the SAME grouped client holdout test set.')
print(f'Random seed: {SEED} | LR: max_iter=1000 | RF: n_estimators=200, max_depth=8')

COMPARISON TABLE: Baseline Rule vs Logistic Regression vs Random Forest
Test set: 12,069 pages | Base rate (is_opportunity): 0.2053 (20.5%)

Metric                Base Rate  Rule Baseline   Logistic Reg  Random Forest
----------------------------------------------------------------------------
Precision@10             20.5%        100.0%         80.0%         60.0%
Precision@20             20.5%        100.0%         85.0%         55.0%
Precision@50             20.5%        100.0%         60.0%         50.0%
Precision@100            20.5%        100.0%         54.0%         42.0%
Precision@200            20.5%        100.0%         53.0%         38.0%
Precision@500            20.5%        100.0%         49.2%         36.6%
ROC AUC                   0.500          0.814          0.750          0.794

Note: All models evaluated on the SAME grouped client holdout test set.
Random seed: 42 | LR: max_iter=1000 | RF: n_estimators=200, max_depth=8


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

---

### Feature Importances

Both Gini (built-in) and Permutation importance **agree on the top 3 features**:
1. `log_impressions` -- Search visibility is the dominant signal (Gini: 0.75, Perm: 0.13)
2. `avg_position` -- SERP rank matters but less than raw traffic volume
3. `engagement_rate` -- GA4 engagement provides complementary behavioral signal

`word_count` and `has_ga4_data` contribute minimally. No feature is suspiciously perfect (which would indicate leakage).

### Error Analysis

Overall test accuracy: **73.7%** (8,890 / 12,069 pages correct).

- **`pos_51_plus`** (deep positions): 99.3% accuracy -- easy, no opportunities exist this deep.
- **`pos_4_10`** (Page 1): 67.6% accuracy -- hardest tier. 25.4% false positive rate. The model over-predicts opportunities on Page 1 because high-impression Page 1 pages look similar whether they are above or below tier median CTR.
- **`pos_1_3`** (Top 3): 67.2% accuracy -- small sample (n=174), highest false positive rate (19.5%).

### Why the Model Struggles

The model cannot see `ctr_gap` (it leaks), so it must guess whether a page's CTR is above or below its tier median using only impressions, position, engagement, word count, and GA4 status. Two pages with identical feature profiles can have opposite labels depending on CTR -- information the honest model simply does not have.

In [7]:
# == Cell 3: Feature Importances (Built-in + Permutation) ==

# ---- Built-in (Gini/MDI) importance ----
print('=' * 70)
print('FEATURE IMPORTANCE: Random Forest Built-in (Gini/MDI)')
print('=' * 70)
gini_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
for feat, imp in gini_imp.items():
    bar = '#' * int(imp * 50)
    print(f'  {feat:<20s} {imp:.4f}  {bar}')

print()

# ---- Permutation importance (on test set) ----
print('=' * 70)
print('FEATURE IMPORTANCE: Permutation Importance (on test set)')
print('=' * 70)
perm_result = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=SEED, n_jobs=-1)
perm_imp = pd.Series(perm_result.importances_mean, index=FEATURES).sort_values(ascending=False)
perm_std = pd.Series(perm_result.importances_std, index=FEATURES)

for feat in perm_imp.index:
    bar = '#' * int(perm_imp[feat] * 200)
    print(f'  {feat:<20s} {perm_imp[feat]:.4f} (+/- {perm_std[feat]:.4f})  {bar}')

print()

# ---- Agreement check ----
gini_top3 = list(gini_imp.index[:3])
perm_top3 = list(perm_imp.index[:3])
if set(gini_top3) == set(perm_top3):
    print(f'Agreement: Both methods agree on the top 3 features: {gini_top3}')
else:
    print(f'Disagreement detected!')
    print(f'  Gini top 3: {gini_top3}')
    print(f'  Perm top 3: {perm_top3}')
    print(f'  This is worth investigating -- Gini can overweight continuous features.')

FEATURE IMPORTANCE: Random Forest Built-in (Gini/MDI)
  log_impressions      0.7482  #####################################
  avg_position         0.1486  #######
  engagement_rate      0.0608  ###
  word_count           0.0286  #
  has_ga4_data         0.0138  

FEATURE IMPORTANCE: Permutation Importance (on test set)
  log_impressions      0.1279 (+/- 0.0023)  #########################
  engagement_rate      0.0611 (+/- 0.0028)  ############
  avg_position         0.0135 (+/- 0.0023)  ##
  word_count           0.0031 (+/- 0.0013)  
  has_ga4_data         -0.0033 (+/- 0.0009)  

Agreement: Both methods agree on the top 3 features: ['log_impressions', 'avg_position', 'engagement_rate']


In [8]:
# == Cell 4: Error Analysis + 3 Concrete Wrong Cases ==

rf_preds = rf.predict(X_test)

# ---- Error rates by position tier ----
print('=' * 80)
print('ERROR ANALYSIS: False Positive / False Negative Rates by Position Tier')
print('=' * 80)

df_err = df_test.copy()
df_err['rf_pred'] = rf_preds
df_err['rf_prob'] = rf_probs
df_err['correct'] = (df_err['rf_pred'] == df_err['is_opportunity']).astype(int)
df_err['false_positive'] = ((df_err['rf_pred'] == 1) & (df_err['is_opportunity'] == 0)).astype(int)
df_err['false_negative'] = ((df_err['rf_pred'] == 0) & (df_err['is_opportunity'] == 1)).astype(int)

tier_order = ['pos_1_3', 'pos_4_10', 'pos_11_20', 'pos_21_50', 'pos_51_plus']
err_table = df_err.groupby('position_tier').agg(
    n=('content_hash_id', 'count'),
    accuracy=('correct', 'mean'),
    false_pos_rate=('false_positive', 'mean'),
    false_neg_rate=('false_negative', 'mean'),
    actual_opp_rate=('is_opportunity', 'mean')
).reindex(tier_order).fillna(0)
print(err_table.round(4).to_string())

# ---- 3 Concrete Wrong Cases ----
print()
print('=' * 80)
print('3 CONCRETE WRONG CASES (Random Forest)')
print('=' * 80)

# False Positives: model said opportunity, actually not
fp = df_err[(df_err['false_positive'] == 1)].sort_values('rf_prob', ascending=False)
# False Negatives: model said not opportunity, actually is
fn = df_err[(df_err['false_negative'] == 1)].sort_values('rf_prob', ascending=True)

case_num = 0
for label, subset, error_type in [('FALSE POSITIVE', fp, 'Model predicted opportunity, but page is NOT one'),
                                   ('FALSE NEGATIVE', fn, 'Model predicted NOT opportunity, but page IS one')]:
    for _, row in subset.head(2 if label == 'FALSE POSITIVE' else 1).iterrows():
        case_num += 1
        print(f'\nCase {case_num} ({label}):')
        print(f'  Error type: {error_type}')
        print(f'  RF probability: {row["rf_prob"]:.3f} | Predicted: {int(row["rf_pred"])} | Actual: {int(row["is_opportunity"])}')
        print(f'  Position tier: {row["position_tier"]} | Avg position: {row["avg_position"]:.1f}')
        print(f'  Impressions: {row["total_impressions"]:,.0f} | CTR: {row["observed_ctr"]:.2f}% | CTR gap: {row["ctr_gap"]:.3f}')
        print(f'  Word count: {row["word_count"]:,.0f} | Engagement rate: {row["engagement_rate"]:.1f}% | GA4: {int(row["has_ga4_data"])}')
        # Explain why it's hard
        if label == 'FALSE POSITIVE':
            print(f'  Why hard: Page looks like an opportunity (high impressions, reasonable position) but its CTR')
            print(f'            is actually at or above tier median, so there is no gap to close.')
        else:
            print(f'  Why hard: Page IS a real opportunity, but the model did not detect it -- likely because')
            print(f'            its feature profile (position, impressions, engagement) resembles non-opportunity pages.')

print()
total_fp = fp.shape[0]
total_fn = fn.shape[0]
total_correct = df_err['correct'].sum()
print(f'Overall test accuracy: {total_correct}/{len(df_err)} ({total_correct/len(df_err)*100:.1f}%)')
print(f'Total false positives: {total_fp} | Total false negatives: {total_fn}')

ERROR ANALYSIS: False Positive / False Negative Rates by Position Tier
                  n  accuracy  false_pos_rate  false_neg_rate  actual_opp_rate
position_tier                                                                 
pos_1_3         174    0.6609          0.2069          0.1322           0.2759
pos_4_10       4266    0.6739          0.2588          0.0673           0.1960
pos_11_20      3860    0.7694          0.1622          0.0684           0.1738
pos_21_50      3629    0.7625          0.1364          0.1011           0.2543
pos_51_plus     140    0.9857          0.0143          0.0000           0.0000

3 CONCRETE WRONG CASES (Random Forest)

Case 1 (FALSE POSITIVE):
  Error type: Model predicted opportunity, but page is NOT one
  RF probability: 0.886 | Predicted: 1 | Actual: 0
  Position tier: pos_4_10 | Avg position: 7.6
  Impressions: 22,978 | CTR: 1.08% | CTR gap: -0.726
  Word count: 3,066 | Engagement rate: 0.0% | GA4: 0
  Why hard: Page looks like an opportunity (

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.